In [1]:
import json
import ollama
from dataclasses import dataclass


@dataclass
class Config:
    llm_model: str = "llama3.1:8b"
    ollama_host: str = "http://localhost:11434"
    temperature: float = 0.0


config = Config()
client = ollama.Client(host=config.ollama_host)


def llm_json(prompt, system=None):
    """Llama al LLM forzando salida JSON."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat(
        model=config.llm_model, messages=messages,
        format="json", options={"temperature": config.temperature},
    )
    raw = resp["message"]["content"]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start: end + 1])


def llm_text(prompt, system=None):
    """Llama al LLM para respuesta en texto libre."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat(
        model=config.llm_model, messages=messages,
        options={"temperature": config.temperature},
    )
    return resp["message"]["content"].strip()


# Sanity check
try:
    client.list()
    print("Ollama conectado en", config.ollama_host)
    print(f"Modelo: {config.llm_model}")
except Exception as e:
    print("Ollama no responde:", e)

Ollama conectado en http://localhost:11434
Modelo: llama3.1:8b


In [2]:
# Extractor de entidades

ENTITY_EXTRACTOR_PROMPT = """You are an entity extractor for a long-term memory system.
The speaker of the conversation snippet is: {speaker}

IMPORTANT: Any first-person pronoun (I, me, my, mine, we, our) refers to {speaker}.
You MUST include {speaker} as an entity whenever the speaker talks about themselves.

Given the conversation snippet, identify the key entities mentioned.
For each entity, output:
- "name": canonical lowercase identifier with underscores (e.g. "san_francisco", "alice").
- "type": one of [person, location, organization, object, concept, event, date, attribute].

OUTPUT FORMAT - return ONLY this JSON, nothing else:
{{"entities": [{{"name": "<name>", "type": "<type>"}}, ...]}}

RULES
- Always include {speaker} (type: person) when first-person pronouns appear.
- Resolve "my partner/husband/wife/kid/etc." to the named person if given.
- Focus on entities that are semantically important, unique, and persistent.
- Ignore generic words. Prefer specific entities.
- If nothing relevant, return {{"entities": []}}.

CONVERSATION:
---
{conversation}
---
"""


def extract_entities(text, speaker):
    """Extrae entidades del texto. Retorna lista de dicts con name y type."""
    prompt = ENTITY_EXTRACTOR_PROMPT.format(conversation=text, speaker=speaker)
    result = llm_json(prompt)
    return result.get("entities", [])

In [3]:
# Extractor de resumen y tópicos
# 
SUMMARY_PROMPT = """You are processing a conversation turn from a user.
Extract two things:
- "summary": ONE short sentence (max 20 words) summarizing what the speaker said.
- "topics": 2-5 short noun phrases capturing the key topics of the turn.

Speaker: {speaker}

OUTPUT FORMAT - return ONLY this JSON, nothing else:
{{"summary": "<one sentence>", "topics": ["<topic1>", "<topic2>", ...]}}

RULES
- Summary should be third-person ("the user mentioned...", "the user said...").
- Topics: short, lowercase noun phrases (e.g. "coffee shop", "moving to new york").
- Do not invent information not present in the turn.

CONVERSATION TURN:
---
{turn}
---
"""


def extract_summary_topics(text, speaker):
    """Extrae resumen breve + tópicos clave del turno."""
    prompt = SUMMARY_PROMPT.format(turn=text, speaker=speaker)
    result = llm_json(prompt)
    return {
        "summary": result.get("summary", text[:80]),
        "topics": result.get("topics", []),
    }

In [4]:
# FASE 1

def phase1_extract_c1(text, speaker):
    """Fase 1 (Opción C.1): solo entidades + summary + topics.
    NO extrae relaciones — eso es lo que diferencia de Mem0g.
    """
    entities = extract_entities(text, speaker=speaker)
    # Garantizar que el speaker esté como entidad
    if not any(e["name"] == speaker for e in entities):
        entities.append({"name": speaker, "type": "person"})
    
    summary_topics = extract_summary_topics(text, speaker=speaker)
    
    return {
        "entities": entities,
        "summary": summary_topics["summary"],
        "topics": summary_topics["topics"],
    }


print("Fase 1 lista. Funciones disponibles:")
print("  - extract_entities(text, speaker)")
print("  - extract_summary_topics(text, speaker)")
print("  - phase1_extract_c1(text, speaker)")

Fase 1 lista. Funciones disponibles:
  - extract_entities(text, speaker)
  - extract_summary_topics(text, speaker)
  - phase1_extract_c1(text, speaker)


In [5]:
test_turns = [
    "I live in San Francisco with my partner Sam.",
    "My favorite coffee shop is Sightglass and I go there every morning before work.",
    "I just moved from Seattle to New York for a new job at Stripe as a software engineer.",
    "Yesterday I went to a LGBTQ support group and it was really powerful.",
    "I have two kids, Ava and Noah, and they keep me busy.",
]

for i, turn in enumerate(test_turns, 1):
    print(f"\n{'='*72}")
    print(f"TURNO {i}: {turn}")
    print('='*72)
    
    result = phase1_extract_c1(turn, speaker="caroline")
    
    print(f"\n  RESUMEN:  {result['summary']}")
    print(f"  TÓPICOS:  {result['topics']}")
    print(f"\n  ENTIDADES ({len(result['entities'])}):")
    for e in result['entities']:
        print(f"    - {e['name']:30s} ({e['type']})")


TURNO 1: I live in San Francisco with my partner Sam.

  RESUMEN:  The user mentioned living in San Francisco with their partner.
  TÓPICOS:  ['san francisco', 'partner']

  ENTIDADES (3):
    - caroline                       (person)
    - san_francisco                  (location)
    - sam                            (person)

TURNO 2: My favorite coffee shop is Sightglass and I go there every morning before work.

  RESUMEN:  The user mentioned their favorite coffee shop.
  TÓPICOS:  ['sightglass', 'coffee shop']

  ENTIDADES (2):
    - caroline                       (person)
    - sightglass                     (organization)

TURNO 3: I just moved from Seattle to New York for a new job at Stripe as a software engineer.

  RESUMEN:  The user mentioned moving to New York for a new job.
  TÓPICOS:  ['moving to new york', 'new job', 'stripe']

  ENTIDADES (4):
    - caroline                       (person)
    - seattle                        (location)
    - new_york                  

In [6]:
import math
import networkx as nx
from datetime import datetime, timezone


def now_iso():
    return datetime.now(timezone.utc).isoformat()


class ConvMemoryGraph:
    """Grafo de memoria conversacional bipartito (Propuesta C.1).
    
    Estructura:
      V = V_turn ∪ V_entity
      E = E_mentions  (turn → entity, único tipo de arista)
    
    Nodos de turno: position, summary, topics, r, role
    Nodos de entidad: entity_type, attributes, first_seen, last_seen, is_speaker
    """
    
    def __init__(self,
                 alpha=0.3, beta=0.4, gamma=0.3, lam=0.3, n_max=30,
                 exclude_speaker_from_relevance=True):
        self.g = nx.MultiDiGraph()
        self.turn_counter = 0
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lam = lam
        self.n_max = n_max

        self.exclude_speaker_from_relevance = exclude_speaker_from_relevance
    
    # Fase 2
    
    def add(self, text, speaker="user"):
        """Ingiere un turno: Fase 1 + Fase 2."""
        # FASE 1
        extracted = phase1_extract_c1(text, speaker)
        entities = extracted["entities"]
        summary = extracted["summary"]
        topics = extracted["topics"]
        
        # 2.1: insertar nodo de turno
        position = self.turn_counter
        turn_id = f"t{position}"
        self.g.add_node(
            turn_id,
            node_type="turn",
            position=position,
            summary=summary,
            topics=topics,
            role="user",
            r=1.0,
            created_at=now_iso(),
        )
        
        #añadir/actualizar entidades + MENTIONS
        for e in entities:
            name = e["name"]
            etype = e["type"]
            if name not in self.g.nodes:
                self.g.add_node(
                    name,
                    node_type="entity",
                    entity_type=etype,
                    attributes={},
                    first_seen=position,
                    last_seen=position,
                    is_speaker=(name == speaker),
                    created_at=now_iso(),
                )
            else:
                self.g.nodes[name]["last_seen"] = position
            
            self.g.add_edge(turn_id, name,
                            edge_type="MENTIONS",
                            created_at=now_iso())
        
        # recomputar r(t_i) para todos los turnos
        t_actual = position
        for tid in self._turn_ids():
            self.g.nodes[tid]["r"] = self._compute_r(tid, t_actual)
        
        # poda condicional
        n_pruned = 0
        if self._n_turns() > self.n_max:
            n_pruned = self._prune_low_relevance_turns()
        
        self.turn_counter += 1
        return {
            "turn_id": turn_id,
            "position": position,
            "n_entities": len(entities),
            "n_pruned": n_pruned,
        }
    
    # r(t_i) 
    
    def _compute_r(self, turn_id, t_actual):
        """Calcula r(t_i) = α·ant(t_i) + β·men(t_i) + γ·ult(t_i)."""
        i = self.g.nodes[turn_id]["position"]
        
        #  antigüedad 
        ant = math.exp(-self.lam * (t_actual - i))
        
        # menciones 
        entities_i = self._entities_of_turn(turn_id)
        sharing_count = 0
        last_mention_j = i
        
        for tid in self._turn_ids():
            j = self.g.nodes[tid]["position"]
            if j < i:
                continue
            if j == i:
                sharing_count += 1  # auto-mención
                continue
            entities_j = self._entities_of_turn(tid)
            if entities_i & entities_j:
                sharing_count += 1
                last_mention_j = max(last_mention_j, j)
        
        men = sharing_count / (t_actual - i + 1)
        
        # recencia
        ult = math.exp(-self.lam * (t_actual - last_mention_j))
        
        return self.alpha * ant + self.beta * men + self.gamma * ult
    

    
    def _turn_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "turn"]
    
    def _entity_ids(self):
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "entity"]
    
    def _n_turns(self):
        return len(self._turn_ids())
    
    def _entities_of_turn(self, turn_id, exclude_speaker=None):
        if exclude_speaker is None:
            exclude_speaker = self.exclude_speaker_from_relevance
        out = set()
        for _, v, d in self.g.out_edges(turn_id, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            if exclude_speaker and self.g.nodes[v].get("is_speaker"):
                continue
            out.add(v)
        return out
    
    def _prune_low_relevance_turns(self):
        n_to_remove = self._n_turns() - self.n_max
        if n_to_remove <= 0:
            return 0
        turns_sorted = sorted(self._turn_ids(),
                              key=lambda tid: self.g.nodes[tid]["r"])
        for tid in turns_sorted[:n_to_remove]:
            self.g.remove_node(tid)
        return n_to_remove
    
    
    def show_state(self):
        """Imprime el estado del grafo de forma legible."""
        print(f"\n{'='*72}")
        print(f"ESTADO DEL GRAFO  |  turnos: {self._n_turns()}  |  entidades: {len(self._entity_ids())}")
        print('='*72)
        
        print(f"\nNODOS DE TURNO:")
        for tid in sorted(self._turn_ids(),
                          key=lambda x: self.g.nodes[x]["position"]):
            d = self.g.nodes[tid]
            menciona = sorted(self._entities_of_turn(tid, exclude_speaker=False))
            bar = "█" * int(d["r"] * 20) + "·" * (20 - int(d["r"] * 20))
            print(f"  [{tid}] pos={d['position']}  r={d['r']:.3f}  {bar}")
            print(f"        resumen:  {d['summary']}")
            print(f"        tópicos:  {d['topics']}")
            print(f"        menciona: {menciona}")
            print()
        
        print(f"NODOS DE ENTIDAD:")
        for eid in sorted(self._entity_ids()):
            d = self.g.nodes[eid]
            mentioning = sorted([u for u, v, td
                                  in self.g.in_edges(eid, data=True)
                                  if td.get("edge_type") == "MENTIONS"])
            flag = " (speaker)" if d.get("is_speaker") else ""
            print(f"  [{eid}]{flag}  tipo={d['entity_type']}  "
                  f"first=t{d['first_seen']}, last=t{d['last_seen']}  "
                  f"mencionada por: {mentioning}")


print("ConvMemoryGraph lista.")
print("Hiperparámetros default: α=0.3, β=0.4, γ=0.3, λ=0.3, N_max=30")

ConvMemoryGraph lista.
Hiperparámetros default: α=0.3, β=0.4, γ=0.3, λ=0.3, N_max=30


In [7]:
mem = ConvMemoryGraph(alpha=0.3, beta=0.4, gamma=0.3, lam=0.3, n_max=30)

dialog = [
    "I live in San Francisco with my partner Sam.",
    "My favorite coffee shop is Sightglass.",
    "I just moved to New York for a new job at Stripe.",
    "I had dinner with Sam at our favorite restaurant.",
]

print("INGESTA DEL DIÁLOGO\n" + "="*72)
for i, turn in enumerate(dialog):
    print(f"\n>>> Turno {i}: '{turn}'")
    result = mem.add(turn, speaker="caroline")
    print(f"    → {result}")

mem.show_state()

INGESTA DEL DIÁLOGO

>>> Turno 0: 'I live in San Francisco with my partner Sam.'
    → {'turn_id': 't0', 'position': 0, 'n_entities': 3, 'n_pruned': 0}

>>> Turno 1: 'My favorite coffee shop is Sightglass.'
    → {'turn_id': 't1', 'position': 1, 'n_entities': 2, 'n_pruned': 0}

>>> Turno 2: 'I just moved to New York for a new job at Stripe.'
    → {'turn_id': 't2', 'position': 2, 'n_entities': 3, 'n_pruned': 0}

>>> Turno 3: 'I had dinner with Sam at our favorite restaurant.'
    → {'turn_id': 't3', 'position': 3, 'n_entities': 3, 'n_pruned': 0}

ESTADO DEL GRAFO  |  turnos: 4  |  entidades: 7

NODOS DE TURNO:
  [t0] pos=0  r=0.622  ████████████········
        resumen:  The user mentioned living in San Francisco with their partner.
        tópicos:  ['san francisco', 'partner']
        menciona: ['caroline', 'sam', 'san_francisco']

  [t1] pos=1  r=0.463  █████████···········
        resumen:  The user mentioned their favorite coffee shop.
        tópicos:  ['favorite coffee shop', 's

In [8]:
ANSWER_PROMPT_C1 = """You are a memory assistant. Use the memories below to answer the question.

The memories are presented in REVERSE CHRONOLOGICAL ORDER: the FIRST turn shown is the MOST RECENT.

CRITICAL RULES:
- Output ONLY the direct answer. No explanations, no "Note:", no "Based on...".
- When memories give different facts about the same thing (e.g., "lives in X" vs "moved to Y"), 
  the MORE RECENT turn wins. A turn that says "moved to NEW_PLACE" supersedes any older 
  "lives in OLD_PLACE" — the user now lives in NEW_PLACE.
- A turn that says "started working at COMPANY" supersedes an older job mention.
- For factual questions: give just the value (e.g. "New York", "$800", "Sam", "5 times").
- For preference questions: 1-2 short sentences.
- Only say "I don't know" if there's truly no related info.

MEMORIES (MOST RECENT FIRST):
{context}

QUESTION: {query}

ANSWER (concise, no preamble):"""


def select_top_k_turns(mem, k=5):
    """Selecciona los k turnos con mayor r(t_i)."""
    turns_sorted = sorted(
        mem._turn_ids(),
        key=lambda tid: mem.g.nodes[tid]["r"],
        reverse=True,
    )
    return turns_sorted[:k]


def build_context(mem, turn_ids):
    """Serializa los turnos en orden cronológico inverso (más recientes primero)."""
    sorted_by_position = sorted(
        turn_ids,
        key=lambda tid: mem.g.nodes[tid]["position"],
        reverse=True,
    )
    
    lines = []
    for tid in sorted_by_position:
        d = mem.g.nodes[tid]
        lines.append(f"[Turn {d['position']}, r={d['r']:.2f}] {d['summary']}")
        if d.get("topics"):
            lines.append(f"  Topics: {', '.join(d['topics'])}")
        mentions = sorted(mem._entities_of_turn(tid, exclude_speaker=False))
        if mentions:
            ents_str = ", ".join(
                f"{m} ({mem.g.nodes[m].get('entity_type', '?')})"
                for m in mentions
            )
            lines.append(f"  Mentions: {ents_str}")
        lines.append("")
    return "\n".join(lines).strip()


def answer(mem, query, k=5, verbose=False):
    """Genera respuesta a una consulta usando top-k turnos por r(t_i)."""
    top_turns = select_top_k_turns(mem, k=k)
    if not top_turns:
        return "I don't know — no memories available yet."
    
    context = build_context(mem, top_turns)
    prompt = ANSWER_PROMPT_C1.format(context=context, query=query)
    
    if verbose:
        print("=== CONTEXTO ENVIADO AL LLM ===")
        print(context)
        print(f"\nQUESTION: {query}\n")
        print("=" * 50)
    
    return llm_text(prompt)


print("Fase 3 lista. Funciones disponibles:")
print("  - select_top_k_turns(mem, k=5)")
print("  - build_context(mem, turn_ids)")
print("  - answer(mem, query, k=5, verbose=False)")

Fase 3 lista. Funciones disponibles:
  - select_top_k_turns(mem, k=5)
  - build_context(mem, turn_ids)
  - answer(mem, query, k=5, verbose=False)


In [9]:
queries = [
    "Where does the user live now?",
    "Who is the user's partner?",
    "Where does the user work?",
    "What is the user's favorite coffee shop?",
    "Where did the user have dinner recently?",
]

for q in queries:
    print(f"\n{'='*72}")
    print(f"Q: {q}")
    print('='*72)
    ans = answer(mem, q, k=5)
    print(f"A: {ans}")


Q: Where does the user live now?
A: New York.

Q: Who is the user's partner?
A: Caroline.

Q: Where does the user work?
A: New York

Q: What is the user's favorite coffee shop?
A: Sightglass.

Q: Where did the user have dinner recently?
A: our_favorite_restaurant


In [10]:
ans = answer(mem, "Where does the user live now?", k=5, verbose=True)
print(f"\nRESPUESTA FINAL: {ans}")

=== CONTEXTO ENVIADO AL LLM ===
[Turn 3, r=1.00] The user mentioned having dinner with someone.
  Topics: dinner, restaurant
  Mentions: caroline (person), our_favorite_restaurant (location), sam (person)

[Turn 2, r=0.64] The user mentioned moving to New York for a new job.
  Topics: new job, moving to new york
  Mentions: caroline (person), new_york (location), stripe (organization)

[Turn 1, r=0.46] The user mentioned their favorite coffee shop.
  Topics: favorite coffee shop, sightglass
  Mentions: caroline (person), sightglass (organization)

[Turn 0, r=0.62] The user mentioned living in San Francisco with their partner.
  Topics: san francisco, partner
  Mentions: caroline (person), sam (person), san_francisco (location)

QUESTION: Where does the user live now?


RESPUESTA FINAL: New York.
